In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Specific catalog and schema
catalog = "_exponent"
schema = "_bronze_allscripts_tw_works"

results = []

# Get all tables in the schema
tables_df = spark.sql(f"SHOW TABLES IN `{catalog}`.`{schema}`")
tables = [row.tableName for row in tables_df.collect()]

print(f"Found {len(tables)} tables in {catalog}.{schema}\n")

for table in tables:
    try:
        columns_df = spark.sql(f"DESCRIBE TABLE `{catalog}`.`{schema}`.`{table}`")
        
        for row in columns_df.collect():
            # Skip partition info and metadata rows
            if row.col_name and not row.col_name.startswith('#'):
                results.append({
                    'table': table,
                    'column_name': row.col_name,
                    'data_type': row.data_type
                })
    except Exception as e:
        print(f"Error describing {table}: {e}")

# Convert to DataFrame
df_schemas = spark.createDataFrame(results)

# Display results
display(df_schemas)

# Optional: Export to CSV
# df_schemas.toPandas().to_csv("/dbfs/FileStore/bronze_allscripts_tw_works_schemas.csv", index=False)